# FASE 3 — Transformações e Feature Engineering 

Após a limpeza e padronização dos dados, esta etapa tem como objetivo transformar a base em um dataset mais adequado para análises analíticas, Business Intelligence e Data Mining. 
 
Nesta fase serão realizadas transformações estruturais, criação de novas variáveis, discretização, normalização e agregações para enriquecer o dataset final.

In [17]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os

# 1. Carga dos dados (Entrada da Etapa)
df = pd.read_csv('../dados_intermediarios/netflix_clean.csv')

# 1. Transformação da Coluna duration

In [18]:
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([a-zA-Z]+)')

# 2. Estruturação de Categorias Múltiplas

In [19]:
df['qtd_generos'] = df['listed_in'].apply(lambda x: len(x.split(',')) if pd.notna(x) else 0)

# 3. Feature Engineering

In [20]:
# idade_conteudo
ano_atual = datetime.now().year
df['idade_conteudo'] = ano_atual - df['release_year']

# decada_lancamento
df['decada_lancamento'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# mes_adicao e ano_adicao
df['date_added'] = pd.to_datetime(df['date_added'])
df['mes_adicao'] = df['date_added'].dt.month
df['ano_adicao'] = df['date_added'].dt.year

# indicador_filme_serie
df['indicador_filme_serie'] = df['type'].apply(lambda x: 1 if x == 'Movie' else 0)

# 4. Discretização

In [21]:
# Criando faixas manuais para filmes
def rotular_duracao(row):
    if row['type'] == 'TV Show':
        return 'Série (Temporadas)'
    val = row['duration_num']
    if val <= 60: return 'Curto'
    elif val <= 120: return 'Médio'
    else: return 'Longo'

df['faixa_duracao'] = df.apply(rotular_duracao, axis=1)

# Exemplo de discretização por idade do conteúdo
df['categoria_idade'] = pd.qcut(df['idade_conteudo'], q=3, labels=['Novo', 'Intermediário', 'Antigo'])

# 5. Normalização e Padronização

In [22]:
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

filmes_idx = df[df['type'] == 'Movie'].index
df.loc[filmes_idx, 'duration_minmax'] = scaler_minmax.fit_transform(df.loc[filmes_idx, ['duration_num']])

df['idade_standardized'] = scaler_std.fit_transform(df[['idade_conteudo']])

# 6. Agregações

In [23]:
agg_pais = df.groupby('country').size().sort_values(ascending=False).head(10)
agg_tipo = df['type'].value_counts(normalize=True) * 100

print(f"Novas colunas criadas: {df.columns.tolist()[-10:]}")
print(f"\nDistribuição por Tipo (%):\n{agg_tipo}")

os.makedirs('../dados_intermediarios', exist_ok=True)
df.to_csv('../dados_intermediarios/netflix_featured.csv', index=False)
print("\nArquivo 'netflix_featured.csv' gerado com sucesso!")

Novas colunas criadas: ['qtd_generos', 'idade_conteudo', 'decada_lancamento', 'mes_adicao', 'ano_adicao', 'indicador_filme_serie', 'faixa_duracao', 'categoria_idade', 'duration_minmax', 'idade_standardized']

Distribuição por Tipo (%):
type
Movie      69.700762
TV Show    30.299238
Name: proportion, dtype: float64

Arquivo 'netflix_featured.csv' gerado com sucesso!
